In [11]:
import os, json, time

In [12]:
from together import Together
import utils

key_file = 'together-personal.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = Together(
  api_key=API_KEY
)


In [ ]:
from importlib import reload
reload(utils)

In [14]:
model_name = 'qwq'
model_endpoint = utils.model_names_to_endpoints[model_name]
model_endpoint

'Qwen/QwQ-32B'

In [15]:
data_dir = '../data/final_dataset'
long_ans_files = ['certamen_translation_long.json', 
                  'junior_scholarship_translation_long.json', 
                  'prosody_caesura_scansion_english.json',
                  'prosody_caesura_scansion_latin.json',
                  #'prosody_feet_questions_english.json',
                  #'prosody_feet_questions_latin.json'
                  ]
long_ans_files = [os.path.join(data_dir, f) for f in long_ans_files]

file_to_data = {}
for file in long_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

certamen_translation_long.json 928
junior_scholarship_translation_long.json 350
prosody_caesura_scansion_english.json 41
prosody_caesura_scansion_latin.json 41


In [16]:
def construct_long_ans_user_prompt(q_dict):
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    question_text += '\n' + utils.long_ans_format_instructions

    return question_text

In [7]:
prompt = construct_long_ans_user_prompt(file_to_data['certamen_translation_long.json'][0])
prompt

'Translate the motto of Alabama: Audēmus iūra nostra dēfendere.\nAt the end of your response, give your answer as:\nAnswer: answer text'

In [8]:
response = client.chat.completions.create(
  model=model_endpoint,
  messages=[
    {
        "role": "system",
        "content": utils.sys_prompt
    },
    {
      "role": "user",
      "content": prompt
    }
  ],
  temperature=0.6, 
  top_p=0.95, 
  #min_p=0,
  #top_k=20
)
print(response.choices[0].message.content)

<think>
Okay, the user wants me to translate the Alabama state motto, "Audēmus iūra nostra dēfendere," into English. Let me start by breaking down each word.

First, "Audēmus" is the first person plural present active indicative of "audeo," which means "we dare" or "we dare to." 

Next, "iūra" is the accusative plural of "iūs," so that's "laws" or "rights." 

Then "nostra" is the possessive adjective, meaning "our," and it should agree with "iūra," which is neuter plural, so "nostra" is correct here. 

"Dēfendere" is the infinitive of "defendere," so "to defend." 

Putting it all together, the structure is "We dare to defend our laws/rights." 

Wait, but in Latin, when you have a verb in the subjunctive after another verb of endeavoring or daring, sometimes the infinitive is used. Here, "audēre" (to dare) is followed by the infinitive, which is "dēfendere." So the full phrase is "We dare to defend our laws (or rights)." 

I should check if "iūra" is better translated as "laws" or "righ

In [17]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [ ]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    if os.path.exists(save_file):
        with open(save_file, 'r') as f:
            q_id_to_resp = json.load(f)
        
    for q_dict in data:
        q_id = q_dict['question_id']
        if q_id in q_id_to_resp:
            i += 1
            continue
        prompt = construct_long_ans_user_prompt(q_dict)

        response = client.chat.completions.create(
            model=model_endpoint,
            messages=[
                {
                    "role": "system",
                    "content": utils.sys_prompt
                },
                {
                "role": "user",
                "content": prompt
                }
            ],
            temperature=0.6, 
            top_p=0.95, 
            #min_p=0,
            #top_k=20
        )
        try:
            resp = response.choices[0].message.content
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.1)
        

        if i % 10 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)

certamen_translation_long.json
junior_scholarship_translation_long.json
prosody_caesura_scansion_english.json
prosody_caesura_scansion_latin.json
  0 / 41


KeyboardInterrupt: 